# Predicting Online Purchase Intention from Web Session Behaviour

This notebook investigates whether e-commerce browsing sessions can be classified as purchase or non-purchase sessions using supervised machine learning.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

RANDOM_STATE = 42

## Initial data examination

In [ ]:
df = pd.read_csv("../data/online_shoppers_intention.csv")
df.head(10)

The first rows show that each record represents one online shopping session. The target variable is `Revenue`, which shows whether the session ended in a purchase.

The input features describe browsing behaviour, timing and visitor/session information. These include page counts, page durations, rates such as `BounceRates` and `ExitRates`, `PageValues`, `Month`, `VisitorType`, `Weekend` and coded technical features such as `Browser`, `Region` and `TrafficType`.

In [ ]:
df.info()

The `df.info()` output shows that the dataset has 12,330 rows and 18 columns. All columns have 12,330 non-null values, so there are no missing values shown at this stage.

The data contains a mixture of integers, floats, strings and Boolean values. This means preprocessing will be needed before modelling, especially for categorical variables and Boolean values.

In [ ]:
summary_checks = pd.DataFrame({
    "rows": [df.shape[0]],
    "columns": [df.shape[1]],
    "duplicate_rows": [df.duplicated().sum()],
    "missing_values_total": [df.isnull().sum().sum()]
})

summary_checks

This summary confirms that the dataset contains 12,330 rows and 18 columns. It also shows that there are no missing values, so no imputation is required before modelling.

There are 125 exact duplicate rows. Since the data represents web sessions, these duplicates may be genuine sessions with the same recorded behaviour rather than data entry errors. I will therefore note this as a possible data issue, but I will not remove them automatically at this stage.

In [ ]:
revenue_summary = pd.DataFrame({
    "count": df["Revenue"].value_counts(),
    "percentage": df["Revenue"].value_counts(normalize=True) * 100
})

revenue_summary

The `Revenue` summary shows that the dataset is imbalanced, with many more non-purchase sessions than purchase sessions. This is realistic for e-commerce data, because most browsing sessions do not end in a transaction.

This affects the modelling stage because accuracy alone may be misleading. A model could perform well on accuracy by mostly predicting the majority class. For this reason, I will later use precision, recall, F1-score and the confusion matrix as well as accuracy.

Next, I examine the numerical feature distributions to understand the scale and spread of the input variables.

In [ ]:
df.describe()

The descriptive statistics show that the numerical features have different ranges. For example, duration and page-count features can have much larger values than rate-based features such as `BounceRates` and `ExitRates`.

This suggests that scaling may be useful for models that are sensitive to feature size, such as logistic regression or k-nearest neighbours. Tree-based models are less affected by this, but the difference in feature ranges is still useful to understand before modelling.

In [ ]:
categorical_cols = ["Month", "VisitorType", "Weekend", "OperatingSystems", "Browser", "Region", "TrafficType"]

categorical_summary = pd.DataFrame({
    "unique_values": df[categorical_cols].nunique(),
    "most_common_value": df[categorical_cols].mode().iloc[0],
    "most_common_count": [df[col].value_counts().iloc[0] for col in categorical_cols]
})

categorical_summary

This summary shows how many unique values each categorical or coded feature contains, and which value appears most often. This is more useful than printing every category count because it gives a quick view of how complex each categorical feature is.

The results confirm that several features need categorical encoding before modelling. This includes text columns such as `Month` and `VisitorType`, as well as coded category columns such as `OperatingSystems`, `Browser`, `Region` and `TrafficType`.

In [ ]:
df.groupby("Revenue")[["PageValues", "BounceRates", "ExitRates", "ProductRelated", "ProductRelated_Duration"]].mean()

The grouped averages show clear differences between purchase and non-purchase sessions. Purchase sessions have much higher average `PageValues`, more `ProductRelated` page visits, longer `ProductRelated_Duration`, lower `BounceRates` and lower `ExitRates`.

This suggests that browsing depth and page value are likely to be useful predictors. These patterns will be tested more formally later through feature importance and model evaluation.